# 🎓 Análisis Exploratorio: ¿Dónde estudian los niños de Barcelona? (cursos 2020/21 – 2024/25)

## Objetivo
**¿Dónde estudian los niños de Barcelona —en la escuela pública, concertada o privada— y depende del distrito en el que viven?**

### Preguntas
1. ¿Qué proporción de alumnos estudia en cada tipo de centro y cómo varía entre distritos?
2. ¿Influye el nivel de renta del distrito en el tipo de escuela?
3. ¿Hay diferencias en el número de alumnos por grupo entre la pública, la concertada y la privada?
4. ¿Cómo ha evolucionado la matrícula en P3? ¿Se nota la caída de la natalidad?
5. ¿Quiénes son los titulares de los centros concertados (órdenes religiosas, fundaciones, cooperativas...)?
6. ¿Hay diferencias en el equilibrio entre chicas y chicos según el tipo de centro?

**Fuente:** Generalitat de Catalunya – *Alumnes matriculats per ensenyament i unitats dels centres docents* (API Socrata)

## 🛠️ Paso 0: Configuración inicial

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 📥 Paso 1: Cargar los datos desde la API
Pedimos a la API solo los registros de Barcelona y de los últimos 5 cursos completos.

In [3]:
URL = "https://analisi.transparenciacatalunya.cat/resource/xvme-26kg.json"
CURSOS = ["2020/2021", "2021/2022", "2022/2023", "2023/2024", "2024/2025"]
LIMITE = 10000

lista_cursos = ",".join(f"'{c}'" for c in CURSOS)
filtro = f"nom_municipi='Barcelona' AND curs IN ({lista_cursos})"

paginas = []
offset = 0
while True:
    params = {
        "$where": filtro,
        "$limit": LIMITE,
        "$offset": offset,
        "$order": ":id",
    }
    respuesta = requests.get(URL, params=params)
    respuesta.raise_for_status()
    datos = respuesta.json()
    if not datos:
        break
    paginas.extend(datos)
    print(f"Descargadas {len(paginas)} filas...")
    offset += LIMITE

df = pd.DataFrame(paginas)
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

Descargadas 10000 filas...
Descargadas 20000 filas...
Descargadas 30000 filas...
Descargadas 40000 filas...
Descargadas 42611 filas...
Filas: 42611, Columnas: 35


### Guardar una copia local
Guardamos los datos en `data/raw/` para no tener que pedirlos a la API cada vez que ejecutemos el notebook.

In [4]:
df.to_csv("../data/raw/matriculas_bcn_raw.csv", index=False)
print("Guardado en data/raw/")

Guardado en data/raw/


## 👀 Paso 2: Primera exploración

### 2.1 Primeras filas

In [6]:
df.head()

,curs,any,codi_centre,denominaci_completa,codi_naturalesa,nom_naturalesa,codi_titularitat,nom_titularitat,codi_delegaci,nom_delegaci,...,nivell,modalitat,per_ode_matr_cula,matr_cules_total,matr_cules_dones,matr_cules_homes,unitats,matr_cula_concertada,tem_tica,grau
0,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,ANUAL,16,4,12,1,NaN,NaN,NaN
1,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,2,PRESENCIAL,ANUAL,18,7,11,1,NaN,NaN,NaN
2,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,3,PRESENCIAL,ANUAL,18,7,11,1,NaN,NaN,NaN
3,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,ANUAL,20,12,8,1,NaN,NaN,NaN
4,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,2,PRESENCIAL,ANUAL,21,10,11,1,NaN,NaN,NaN


### 2.2 Dimensiones del dataset

In [7]:
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

Filas: 42611, Columnas: 35


### 2.3 Información general: columnas, tipos y valores no nulos

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42611 entries, 0 to 42610
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   curs                      42611 non-null  str  
 1   any                       42611 non-null  str  
 2   codi_centre               42611 non-null  str  
 3   denominaci_completa       42611 non-null  str  
 4   codi_naturalesa           42611 non-null  str  
 5   nom_naturalesa            42611 non-null  str  
 6   codi_titularitat          42611 non-null  str  
 7   nom_titularitat           42611 non-null  str  
 8   codi_delegaci             42611 non-null  str  
 9   nom_delegaci              42611 non-null  str  
 10  codi_comarca              42611 non-null  str  
 11  nom_comarca               42611 non-null  str  
 12  codi_municipi_5           42611 non-null  str  
 13  codi_municip_6            42611 non-null  str  
 14  nom_municipi              42611 non-null  str  
 

### 2.4 Valores faltantes por columna

In [9]:
faltantes = df.isnull().sum()
porcentaje = (faltantes / len(df) * 100).round(1)
pd.DataFrame({"faltantes": faltantes, "%": porcentaje})[faltantes > 0].sort_values("%", ascending=False)

,faltantes,%
tem_tica,34952,82.0
grau,29592,69.4
matr_cula_concertada,22486,52.8
nivell,1536,3.6
codi_ensenyament,481,1.1
nom_ensenyament,482,1.1
coordenades_utm_x,30,0.1
coordenades_utm_y,30,0.1
coordenades_geo_x,30,0.1
coordenades_geo_y,30,0.1


### 2.5 Filas duplicadas

In [10]:
print(f"Filas duplicadas: {df.duplicated().sum()}")

Filas duplicadas: 26


In [12]:
df[df.duplicated(keep=False)].sort_values(["codi_centre", "curs"]).head(10)

,curs,any,codi_centre,denominaci_completa,codi_naturalesa,nom_naturalesa,codi_titularitat,nom_titularitat,codi_delegaci,nom_delegaci,...,nivell,modalitat,per_ode_matr_cula,matr_cules_total,matr_cules_dones,matr_cules_homes,unitats,matr_cula_concertada,tem_tica,grau
10062,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,26,18,8,0,NaN,NaN,IDILNA
10105,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,29,19,10,0,NaN,NaN,IDILNB
10125,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,24,15,9,0,NaN,NaN,IDILNB
10128,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,2,PRESENCIAL,2n QUADRIMESTRE,20,14,6,0,NaN,NaN,IDILNB
10146,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,8,3,5,0,NaN,NaN,IDILNI
10162,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,14,10,4,0,NaN,NaN,IDILNI
10180,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,26,18,8,0,NaN,NaN,IDILNA
10196,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,29,19,10,0,NaN,NaN,IDILNB
10200,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,24,15,9,0,NaN,NaN,IDILNB
10201,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,2,PRESENCIAL,2n QUADRIMESTRE,20,14,6,0,NaN,NaN,IDILNB


### 2.6 Valores únicos de los distritos

In [11]:
df["nom_dm"].value_counts()

nom_dm
Eixample                     6824
Sarrià - Sant Gervasi        5715
Sant Martí                   4938
Ciutat vella                 4622
Horta - Guinardó             4229
Nou Barris                   4188
Sants - Montjuïc             3961
Sant Andreu                  3145
Gràcia                       2610
Les Corts                    2314
DM VIII - Nou Barris           19
DM VII - Horta - Guinardó      19
DM II - Eixample                6
DM III - Sants - Montjuïc       6
DM IX - Sant Andreu             6
DM X - Sant Martí               5
DM I - Ciutat vella             4
Name: count, dtype: int64

### 2.7 ¿Los valores faltantes de `matr_cula_concertada` son de centros públicos?

In [13]:
pd.crosstab(df["nom_naturalesa"], df["matr_cula_concertada"].fillna("FALTANTE"))

matr_cula_concertada,FALTANTE,NO,SI
nom_naturalesa,,,
Privat,4096,6099,13046
Públic,18390,980,0


### 📝 Observaciones de la primera exploración
- **Tipos de datos:** todas las columnas llegan como texto (`str`), incluso las numéricas (`matr_cules_total`, `unitats`). Hay que convertirlas.
- **Distritos inconsistentes:** `nom_dm` tiene valores como "DM VIII - Nou Barris" además de "Nou Barris" (17 valores para 10 distritos). Además, "Ciutat vella" debería ser "Ciutat Vella" (importante para el merge con los datos de renta).
- **Concertada:** el 52% de `matr_cula_concertada` está vacío, pero el 82% de esos vacíos (18.390) son centros públicos, donde la pregunta no aplica (*faltante estructural*). El problema real son **4.096 registros privados** sin saber si son concertados o privados. Además, en centros públicos el mismo significado aparece como vacío o como "NO".
- **Duplicados:** 26 filas duplicadas, todas de dos Escoles Oficials d'Idiomes (enseñanza de adultos).
- **Coordenadas:** 30 filas sin coordenadas.
- **Columnas casi vacías:** `tem_tica` y `grau` solo están rellenas en algunos tipos de estudios.

### 🎯 Decisión de alcance
Nuestra pregunta es sobre **los niños**, así que nos quedamos con las etapas de 3 a 16 años:
- **Educació Infantil – segon cicle (P3-P5):** casi universal y se imparte en las escuelas. El primer ciclo (0-3) es voluntario y funciona como un sistema aparte (escoles bressol).
- **Educació Primària y ESO:** etapas obligatorias, todas las familias tienen que elegir centro.

Excluimos Batxillerat y FP (no obligatorios; el concierto suele cubrir solo las etapas obligatorias, lo que distorsionaría la comparación) y la enseñanza de adultos, idiomas y artísticas (fuera de la pregunta). Con este filtro, los 26 duplicados también quedan fuera.

## 🧹 Paso 3: Limpieza de datos

### 3.1 Filtrar las etapas de 3 a 16 años
Aplicamos la decisión de alcance. Usamos `codi_ensenyament` (código) en lugar de `nom_estudis` (texto), porque los códigos son estables y no dependen de acentos ni mayúsculas.

In [14]:
ETAPAS = {
    "EINFLOE 2C": "Infantil (P3-P5)",
    "EPRILOE": "Primaria",
    "ESO LOE": "ESO",
}

filas_antes = len(df)
df_limpio = df[df["codi_ensenyament"].isin(ETAPAS.keys())].copy()
df_limpio["etapa"] = df_limpio["codi_ensenyament"].map(ETAPAS)

print(f"Filas antes: {filas_antes}, después: {len(df_limpio)} ({len(df_limpio) / filas_antes:.0%})")
df_limpio["etapa"].value_counts()

Filas antes: 42611, después: 19289 (45%)


etapa
Primaria            9910
Infantil (P3-P5)    5067
ESO                 4312
Name: count, dtype: int64

### 3.2 Comprobar duplicados tras el filtro

In [15]:
print(f"Filas duplicadas: {df_limpio.duplicated().sum()}")

Filas duplicadas: 0


### 3.3 Seleccionar y renombrar columnas
Nos quedamos solo con las columnas que necesitamos para responder a nuestras preguntas y les damos nombres en español, más fáciles de entender.

**Eliminamos:**
- Columnas con el mismo valor en todas las filas (municipio, comarca, delegación, período de matrícula): como solo analizamos Barcelona, no aportan información.
- Columnas redundantes: códigos que tienen su nombre en otra columna (`codi_naturalesa`), `any` (mismo dato que `curs`) y coordenadas UTM (tenemos latitud/longitud).
- Columnas de tipo de estudios: las sustituye la nueva columna `etapa`.
- `tem_tica` y `grau`: 100% vacías en nuestras etapas.
- `codi_districte_municipal`: mezcla códigos y nombres; usamos `nom_dm`.

In [16]:
COLUMNAS = {
    "curs": "curso",
    "codi_centre": "codigo_centro",
    "denominaci_completa": "nombre_centro",
    "nom_naturalesa": "naturaleza",
    "matr_cula_concertada": "concertada",
    "nom_titularitat": "titularidad",
    "nom_dm": "distrito",
    "coordenades_geo_y": "latitud",
    "coordenades_geo_x": "longitud",
    "nivell": "nivel",
    "modalitat": "modalidad",
    "matr_cules_total": "alumnos_total",
    "matr_cules_dones": "alumnos_mujeres",
    "matr_cules_homes": "alumnos_hombres",
    "unitats": "grupos",
}

df_limpio = df_limpio[list(COLUMNAS.keys()) + ["etapa"]].rename(columns=COLUMNAS)
print(f"Columnas: {df_limpio.shape[1]}")
df_limpio.head()

Columnas: 16


,curso,codigo_centro,nombre_centro,naturaleza,concertada,titularidad,distrito,latitud,longitud,nivel,modalidad,alumnos_total,alumnos_mujeres,alumnos_hombres,grupos,etapa
0,2022/2023,08001595,Escola Baixeras,Públic,NaN,Departament d'Ensenyament,Ciutat vella,41.382837911,2.179781021,1,PRESENCIAL,16,4,12,1,Infantil (P3-P5)
1,2022/2023,08001595,Escola Baixeras,Públic,NaN,Departament d'Ensenyament,Ciutat vella,41.382837911,2.179781021,2,PRESENCIAL,18,7,11,1,Infantil (P3-P5)
2,2022/2023,08001595,Escola Baixeras,Públic,NaN,Departament d'Ensenyament,Ciutat vella,41.382837911,2.179781021,3,PRESENCIAL,18,7,11,1,Infantil (P3-P5)
3,2022/2023,08001595,Escola Baixeras,Públic,NaN,Departament d'Ensenyament,Ciutat vella,41.382837911,2.179781021,1,PRESENCIAL,20,12,8,1,Primaria
4,2022/2023,08001595,Escola Baixeras,Públic,NaN,Departament d'Ensenyament,Ciutat vella,41.382837911,2.179781021,2,PRESENCIAL,21,10,11,1,Primaria


### 3.4 Corregir los tipos de datos
La API devuelve todo como texto. Convertimos a número las columnas numéricas.
- `pd.to_numeric(errors="coerce")`: si algún valor no se puede convertir, lo pone como `NaN` en lugar de dar error. Después comprobamos que no hayamos creado nulos nuevos.
- `Int64` (con mayúscula) es el entero de pandas que admite valores nulos; `nivel` tiene 107 vacíos.

In [17]:
COLUMNAS_ENTERAS = ["nivel", "alumnos_total", "alumnos_mujeres", "alumnos_hombres", "grupos"]
COLUMNAS_DECIMALES = ["latitud", "longitud"]

nulos_antes = df_limpio[COLUMNAS_ENTERAS + COLUMNAS_DECIMALES].isnull().sum()

for col in COLUMNAS_ENTERAS:
    df_limpio[col] = pd.to_numeric(df_limpio[col], errors="coerce").astype("Int64")
for col in COLUMNAS_DECIMALES:
    df_limpio[col] = pd.to_numeric(df_limpio[col], errors="coerce")

nulos_despues = df_limpio[COLUMNAS_ENTERAS + COLUMNAS_DECIMALES].isnull().sum()
print("Nulos nuevos creados por la conversión:", (nulos_despues - nulos_antes).sum())
df_limpio.dtypes

Nulos nuevos creados por la conversión: 0


curso                  str
codigo_centro          str
nombre_centro          str
naturaleza             str
concertada             str
titularidad            str
distrito               str
latitud            float64
longitud           float64
nivel                Int64
modalidad              str
alumnos_total        Int64
alumnos_mujeres      Int64
alumnos_hombres      Int64
grupos               Int64
etapa                  str
dtype: object

### 3.5 Estadística descriptiva de las columnas numéricas

In [18]:
df_limpio.describe()

,latitud,longitud,nivel,alumnos_total,alumnos_mujeres,alumnos_hombres,grupos
count,19263.000000,19263.000000,19182.0,19289.0,19289.0,19289.0,19289.0
mean,41.412515,2.154271,2.875769,46.442947,22.758774,23.684172,1.832029
std,0.063696,0.122262,1.542829,37.159263,18.555144,19.458842,0.96694
min,40.539402,0.476270,1.0,1.0,0.0,0.0,0.0
25%,41.391370,2.143079,2.0,25.0,12.0,12.0,1.0
50%,41.407415,2.166792,3.0,43.0,20.0,21.0,2.0
75%,41.424457,2.182434,4.0,54.0,28.0,29.0,2.0
max,42.395185,3.082864,6.0,1037.0,518.0,531.0,7.0


### 3.6 Coordenadas imposibles
En la estadística descriptiva vimos que la latitud va de 40,54 a 42,40 y la longitud de 0,48 a 3,08. Barcelona está aproximadamente entre **41,3–41,5 de latitud** y **2,0–2,3 de longitud**, así que hay centros situados fuera de la ciudad (cerca de Tortosa o de la frontera con Francia), aunque su distrito es de Barcelona.

**❓ ¿Cuántas filas tienen coordenadas fuera de Barcelona y de qué cursos son?**

In [19]:
fuera_bcn = df_limpio[~df_limpio["latitud"].between(41.3, 41.5) | ~df_limpio["longitud"].between(2.0, 2.3)]
print(f"Filas con coordenadas fuera de Barcelona: {len(fuera_bcn)}")
fuera_bcn["curso"].value_counts()

Filas con coordenadas fuera de Barcelona: 460


curso
2021/2022    386
2022/2023     38
2023/2024     13
2024/2025     13
2020/2021     10
Name: count, dtype: int64

> **💡 ¿Qué aprendemos?** Hay 460 filas con coordenadas no válidas: 434 fuera de Barcelona y 26 sin coordenadas. **386 de ellas (84%) son del curso 2021/2022**, lo que indica un error en la publicación de ese año más que un problema de los centros.

> **✅ Decisión:** la ubicación de un centro no cambia de un año a otro. Por eso sustituimos las coordenadas incorrectas por la mediana de las coordenadas válidas del mismo centro (`codigo_centro`) en otros cursos.
> - **¿Por qué no eliminamos las filas?** Perderíamos alumnos reales de nuestro análisis. Solo la ubicación es incorrecta; el resto de los datos es válido.
> - **¿Por qué no buscamos por el nombre del centro?** Nombres como "Montserrat" o "Sant Pau" pueden corresponder a muchos lugares de Cataluña. Usar los datos del mismo centro en otros años es más fiable.

In [22]:
LAT_BCN = (41.3, 41.5)
LON_BCN = (2.0, 2.3)

def coordenadas_validas(df):
    return df["latitud"].between(*LAT_BCN) & df["longitud"].between(*LON_BCN)

validas = coordenadas_validas(df_limpio)
coords_por_centro = (
    df_limpio[validas]
    .groupby("codigo_centro")[["latitud", "longitud"]]
    .median()
)

df_limpio.loc[~validas, "latitud"] = df_limpio.loc[~validas, "codigo_centro"].map(coords_por_centro["latitud"])
df_limpio.loc[~validas, "longitud"] = df_limpio.loc[~validas, "codigo_centro"].map(coords_por_centro["longitud"])

print(f"Filas con coordenadas no válidas: antes {(~validas).sum()}, después {(~coordenadas_validas(df_limpio)).sum()}")

Filas con coordenadas no válidas: antes 460, después 59


> **📊 Resultado:** pasamos de 460 a 59 filas no válidas. El mismo método también rellenó las 26 filas que no tenían coordenadas (`between()` devuelve `False` para los valores nulos).
>
> Las 59 filas restantes son todas del *Institut Escola Coves d'en Cimany*, que no tiene ninguna coordenada válida en ningún curso.

> **✅ Decisión:** como es un único centro, buscamos su ubicación manualmente en Google Maps.

In [23]:
COVES_DEN_CIMANY = df_limpio["nombre_centro"] == "Institut Escola Coves d'en Cimany"
df_limpio.loc[COVES_DEN_CIMANY, ["latitud", "longitud"]] = [41.420676, 2.152548]

print(f"Filas con coordenadas no válidas: {(~coordenadas_validas(df_limpio)).sum()}")

Filas con coordenadas no válidas: 0


> **📊 Resultado:** 0 filas con coordenadas no válidas. Todos los centros tienen ahora una ubicación dentro de Barcelona, necesaria para el mapa del dashboard.

> 📋 *Para el informe de calidad de datos:* 434 coordenadas fuera de Barcelona (casi todas de 2021/22) y 26 vacías → corregidas con el mismo centro en otros cursos; 1 centro corregido manualmente.

In [20]:
df_limpio.sort_values("alumnos_total", ascending=False).head(8)

,curso,codigo_centro,nombre_centro,naturaleza,concertada,titularidad,distrito,latitud,longitud,nivel,modalidad,alumnos_total,alumnos_mujeres,alumnos_hombres,grupos,etapa
23344,2022/2023,08040126,Lyceé Français de Barcelone,Privat,NO,Altres estats UE,Sarrià - Sant Gervasi,41.392699,2.118257,<NA>,ESTRANGERS,1037,511,526,0,ESO
18634,2021/2022,08040126,Lyceé Français de Barcelone,Privat,NO,Altres estats UE,Sarrià - Sant Gervasi,41.392699,2.118257,<NA>,ESTRANGERS,1032,501,531,0,ESO
18633,2021/2022,08040126,Lyceé Français de Barcelone,Privat,NO,Altres estats UE,Sarrià - Sant Gervasi,41.392699,2.118257,<NA>,ESTRANGERS,1030,518,512,0,Primaria
32083,2023/2024,08040126,Lyceé Français de Barcelone,Privat,NO,Altres estats UE,Sarrià - Sant Gervasi,41.392699,2.118257,<NA>,ESTRANGERS,1027,498,529,0,ESO
41199,2024/2025,08040126,Lyceé Français de Barcelone,Privat,NO,Altres estats UE,Sarrià - Sant Gervasi,41.392760,2.118357,<NA>,ESTRANGERS,1025,496,529,0,ESO
10337,2020/2021,08040126,Lyceé Français de Barcelone,Privat,NO,Altres estats UE,Sarrià - Sant Gervasi,41.392699,2.118257,<NA>,ESTRANGERS,1018,494,524,0,ESO
23343,2022/2023,08040126,Lyceé Français de Barcelone,Privat,NO,Altres estats UE,Sarrià - Sant Gervasi,41.392699,2.118257,<NA>,ESTRANGERS,1012,492,520,0,Primaria
10336,2020/2021,08040126,Lyceé Français de Barcelone,Privat,NO,Altres estats UE,Sarrià - Sant Gervasi,41.392699,2.118257,<NA>,ESTRANGERS,1001,514,487,0,Primaria


In [21]:
df_limpio[df_limpio["grupos"] == 0][["naturaleza", "etapa", "modalidad"]].value_counts()

naturaleza  etapa             modalidad 
Privat      Primaria          PRESENCIAL    135
            Infantil (P3-P5)  PRESENCIAL    127
            ESO               PRESENCIAL     73
            Primaria          ESTRANGERS     39
            Infantil (P3-P5)  ESTRANGERS     38
            ESO               ESTRANGERS     30
Públic      Primaria          PRESENCIAL     10
            Infantil (P3-P5)  PRESENCIAL      3
Name: count, dtype: int64